In [ ]:
# input
filtered_result = "./tmp/filtered_interface.tsv"
high_pred_result = "../predict_afdb/data/pred_ge_3_clique_3.tsv"
pred_type_file = "./tmp/pred_homomer_merged.tsv"
homomer_info = "../_database/homomers_four_species/homomer_four_species.csv"
homomer_struct = "../_database/homomers_four_species/homomer_pdb_files.tsv"
anno_file = "../collect_annotation/afdb_anno/data/entryId-repId-pfamId-tedId-annoLevel-sp.tsv"
# output
pred_info = "./data/pred_info.tsv"

In [2]:
import pandas as pd

In [3]:
df = pd.read_table(filtered_result)
df['seq_id'] = df['seq_id'].map(lambda x: x.split("-")[1])
df["site_max_chain_num"] = df['site_chain_name'].map(lambda x: max([len(set(i.split(","))) for i in x.split(";")]))

df_struct = pd.read_table(homomer_struct)
def get_chain_num_from_path(f: str):
    if "AF_dimer_models_full_length_relaxed" in f:
        return 2
    
    elif "full_complexes_bigbang" in f:
        return int(f.split("/")[-1].split("_")[3][1:])

    else:
        raise ValueError
id2chain_num = dict(zip(df_struct['seq_id'], df_struct['file'].map(lambda x: get_chain_num_from_path(x))))
df['chain_num'] = df['seq_id'].map(lambda x: id2chain_num[x])

In [4]:
df_info = pd.read_table(homomer_info, usecols=['code_sub', 'sym', 'org'])
df_info.rename(columns={"code_sub": "seq_id"}, inplace=True)
df_anno = pd.read_table(anno_file, header=None, names=["seq_id", "rep", "pfam_id", "ted_id", "anno_level", "sp"])

df = pd.merge(df, df_anno, on="seq_id")
df = pd.merge(df, df_info, on="seq_id")

In [5]:
df_type_pred = pd.read_table(pred_type_file)
df = pd.merge(df, df_type_pred, on="seq_id")

In [6]:
df_high_conf = pd.read_table(high_pred_result)
df_high_conf["seq_id"] = df_high_conf["seq_id"].map(lambda x: x.split("-")[1])
df_high_conf["high_conf_pred_seq_num"] = df_high_conf["posi"].map(lambda x: ",".join([str(int(i) + 1) for i in x.split(",")]))
df_high_conf.drop(columns=["plddt", "pred", "site", "posi"], inplace=True)
df = pd.merge(df, df_high_conf, on="seq_id", how="left")

In [7]:
df.to_csv(pred_info, sep="\t", index=None)